# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

*Read `skills/README.md`, then loaded `skills/building-baselines/SKILL.md` and `skills/flyrank/flyrank-data/SKILL.md` as directed on this assignment's card. Lane: Refresh / Content Opportunity Scoring (Lane 2). Working on the starter dataset (`data/raw/content_refresh_anonymized.csv`) — lane confirmed, not switching.*

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

**The rule, in plain words:** a page is worth reviewing first if it's declining and has real visibility (enough impressions to matter) — ranked higher the more impressions it's wasting, with extra priority if it's *also* either moderately stale since its last update, or badly underperforming CTR for how well it's actually ranking.

**Before coding it, checking the two signals it leans on** — per the building-baselines skill, a rule you can't check is a rule you're guessing at.

**Signal 1 — staleness vs. decline (linked to FlyRank's real `stale_visible_page` refresh flag from this week's session):**

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
eligible = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)].copy()
eligible["is_declining_label"] = (eligible["trend_direction"] == "down").astype(int)

bins = [0, 30, 110, 100000]
bucket_labels = ["fresh (<30d)", "moderate (30-110d)", "stale (110d+)"]
eligible["staleness_bucket"] = pd.cut(eligible["days_since_last_update"], bins=bins, labels=bucket_labels, right=False)

staleness_table = eligible.groupby("staleness_bucket", observed=True).agg(
    n=("is_declining_label", "count"),
    declining_rate=("is_declining_label", "mean")
).round(3)

print(staleness_table)
print(f"\nBase rate (all eligible rows): {eligible['is_declining_label'].mean():.3f}")

**Verdict: MIXED.** `n=20,480` fresh / `n=9,290` moderate / `n=230` stale. Decline rate rises from 0.511 (fresh) to 0.613 (moderate) — the direction I expected — but then *drops* to 0.426 in the stale (110d+) bucket, below even the fresh baseline, and that bucket only has 230 rows. This isn't the clean "older = more likely declining" story a refresh rule would want to lean on alone. A clearly mixed result like this is still useful: it tells me staleness alone is a weak, non-monotonic signal here, so I'm using it as a *secondary* priority boost in the rule, never the primary gate.

**Signal 2 — CTR vs. position tier (linked to FlyRank's real CTR-fix / `ctr_review_candidate` logic from this week's session):**

In [ ]:
ctr_table = eligible[eligible["impressions_90d"] >= 100].groupby("position_tier", observed=True).agg(
    n=("ctr", "count"),
    mean_ctr=("ctr", "mean")
).round(4).sort_values("mean_ctr", ascending=False)

print(ctr_table)

**Verdict: CONFIRMED.** CTR drops cleanly and monotonically as position tier worsens — page_1 (n=8,633): 0.355%, top_3 (n=533): 0.334%, striking (n=5,903): 0.256%, page_3_5 (n=6,058): 0.142%, deep (n=879): 0.055%. This is exactly the pattern FlyRank's CTR-fix logic assumes (comparing pages only within their tier), and it holds with real sample sizes at every tier — safe to build on.

**Reason codes the rule can output (one per row, priority order):**
- `low_ctr_visible_page` — CTR under 0.5% at position 20 or better (a real underperformance, per the confirmed signal).
- `moderately_stale_page` — 30–110 days since last update, when low-CTR doesn't already explain it (the mixed signal, used only as a secondary boost).
- `declining_with_demand` — declining and visible, but neither extra condition applies (the base case).

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:
declining_flag = (eligible["trend_direction"] == "down").astype(int)
visible_flag = (eligible["impressions_90d"] >= 500).astype(int)
stale_flag = ((eligible["days_since_last_update"] >= 30) & (eligible["days_since_last_update"] < 110)).astype(int)
low_ctr_flag = ((eligible["ctr"] < 0.5) & (eligible["avg_position"] > 0) & (eligible["avg_position"] <= 20)).astype(int)

# Readable on purpose: no fitted weights, just simple conditions.
eligible["score"] = declining_flag * visible_flag * eligible["impressions_90d"] * (1 + 0.5*stale_flag + 0.5*low_ctr_flag)

def reason_code(low_ctr, stale):
    if low_ctr:
        return "low_ctr_visible_page"
    elif stale:
        return "moderately_stale_page"
    else:
        return "declining_with_demand"

eligible["reason_code"] = [reason_code(l, s) for l, s in zip(low_ctr_flag, stale_flag)]
eligible["action"] = np.where(eligible["score"] > 0, "review_for_refresh", "monitor")

print(f"Flagged for review: {(eligible['action'] == 'review_for_refresh').sum()} out of {len(eligible)}")
print(eligible["reason_code"].value_counts())

In [ ]:
import os

ranked = eligible.sort_values("score", ascending=False).reset_index(drop=True)
output_cols = ["content_id", "client_id", "score", "reason_code", "action",
               "impressions_90d", "ctr", "avg_position", "days_since_last_update", "trend_direction"]

os.makedirs("../outputs", exist_ok=True)
ranked[output_cols].to_csv("../outputs/baseline_action_score.csv", index=False)
print(f"Wrote {len(ranked)} rows to work/outputs/baseline_action_score.csv")
ranked[output_cols].head(10)

## 3. Top-20 review

*For each of your top 20: action, reason code, confidence note, and what would make it wrong.*

In [ ]:
ranked[output_cols].head(20)

**Row-by-row review (real output from the queue above):**

1. `content_5fe46e04994d` — 517,715 impr, CTR 0.14%, pos 4.2, `low_ctr_visible_page`. Strong position, almost no clicks — likely a title/snippet problem. **Wrong if:** the query intent is informational and a SERP feature (featured snippet, PAA) is absorbing clicks regardless of title quality.
2. `content_8c19996aa890` — 509,252 impr, CTR 0.15%, pos 2.5, `low_ctr_visible_page`. Near-top position, still terrible CTR. **Wrong if:** rich results/schema already shipped and this row predates that fix.
3. `content_4c36c775b818` — 463,103 impr, CTR 0.41%, pos 2.3, `low_ctr_visible_page`. Borderline — CTR close to the 0.5% cutoff. **Wrong if:** 0.41% is actually normal for this specific SERP layout (heavy ads/PAA).
4. `content_1a9e894be2e2` — 416,180 impr, CTR 0.23%, pos 4.0, `low_ctr_visible_page`. Clear title/meta candidate. **Wrong if:** the page has since been consolidated/redirected and this row is stale metadata.
5. `content_cb112fce36be` — 309,910 impr, CTR 0.16%, pos 5.6, stale 104d, `low_ctr_visible_page`. Both stale and low-CTR, but coded low-CTR by priority order. **Wrong if:** a refresh already shipped after this snapshot was taken.
6. `content_2c2606c5d176` — 347,399 impr, CTR 0.53% (just above the cutoff), pos 4.2, `moderately_stale_page`. Flagged mainly for staleness despite decent CTR. **Wrong if:** this is reference content where "staleness" doesn't actually hurt performance.
7. `content_9532f197bbc8` — 309,192 impr, **CTR 0.87%** (healthy), pos **2.0** (excellent), `moderately_stale_page`. **This is my weak pick — flagged in Section 4.**
8. `content_c8e9d6ab9013` — 208,678 impr, CTR **0.00%**, pos 9.7, `low_ctr_visible_page`. Zero clicks despite real borderline-page-1 visibility — a strong genuine candidate. **Wrong if:** this is a tracking gap (clicks not recorded) rather than a real zero.
9. `content_3d94572c3a35` — 190,623 impr, CTR 0.24%, pos 4.3, `low_ctr_visible_page`. Standard candidate, no red flags.
10. `content_01908772c6db` — 187,893 impr, CTR 0.45%, pos 4.0, `low_ctr_visible_page`. Borderline CTR, close to cutoff — lower confidence than #1–4.
11. `content_008fb02c46cb` — 236,803 impr, CTR 0.26%, pos 4.4, `low_ctr_visible_page`. Standard candidate.
12. `content_813e88069237` — 233,561 impr, CTR 0.06%, pos **26.2**, `moderately_stale_page` (position over 20 excludes it from the low-CTR rule by design). **Wrong if:** the page-2 position means the CTR is expected, not a fixable snippet problem — this may deserve a ranking push, not a CTR rewrite.
13. `content_cea79ef51519` — 208,798 impr, CTR 0.23%, pos 5.2, `low_ctr_visible_page`. Standard candidate.
14. `content_f42eb861c6dd` — 152,467 impr, CTR 0.13%, pos 6.5, `low_ctr_visible_page`. Standard candidate.
15. `content_11fcfd65d94c` — 149,083 impr, CTR 0.15%, pos 6.2, `low_ctr_visible_page`. Standard candidate.
16. `content_bf7bff5d0756` — 197,199 impr, CTR 0.22%, pos 6.8, `low_ctr_visible_page`. Standard candidate.
17. `content_97a86caf3a3d` — 147,670 impr, CTR 0.07%, pos 6.4, `low_ctr_visible_page`. Very low CTR, strong candidate.
18. `content_cbd93118300b` — 145,292 impr, CTR 0.46%, pos 3.3, `low_ctr_visible_page`. Borderline CTR, lower confidence.
19. `content_9463d30d5826` — 192,478 impr, CTR 0.29%, pos 5.7, `low_ctr_visible_page`. Standard candidate.
20. `content_c1350d507c68` — 142,505 impr, CTR 0.29%, pos 3.9, `low_ctr_visible_page`. Standard candidate.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

**Weak pick found: #7, `content_9532f197bbc8`.** CTR 0.87% and position 2.0 are both genuinely good — there is no real underperformance here. It only made the list because it sits in the 30–110-day staleness window, and Section 1's own signal check already showed staleness is a *mixed*, non-monotonic signal. This is the rule doing exactly what a hand-written rule does when one of its inputs is weak: it surfaces a page that a human reviewer would immediately dismiss. Per the building-baselines skill, finding zero weak picks in a top-20 would mean I wasn't looking hard enough — finding this one is the honest result.

**Also worth flagging: #12, `content_813e88069237`.** Position 26.2 is page-3+, so the "underperforming CTR for its position" framing may not even apply — a 0.06% CTR at that depth might just be normal, not fixable by a title rewrite. This may need a ranking push, not a content refresh — a different action than the label suggests.

In [ ]:
# Leakage check.
used_columns = ["trend_direction", "impressions_90d", "ctr", "avg_position", "days_since_last_update"]
print("Columns used in the rule:", used_columns)

forbidden_terms = ["health_score", "priority_score", "action_type", "refresh_tier", "flag"]
present_forbidden = [c for c in df.columns if any(term in c.lower() for term in forbidden_terms)]
print("Product-decision columns present in the dataset at all:", present_forbidden if present_forbidden else "none — confirmed not shipped, per the flyrank-data skill")

future_window_terms = ["last_30d", "prev_30d"]
used_future = [c for c in used_columns if any(term in c for term in future_window_terms)]
print("Future-window columns used in the rule:", used_future if used_future else "none")

**On `trend_direction`:** the rule uses it directly as the "declining" gate — this is intentional and not leakage at the baseline stage. A baseline rule's whole job is to encode a human decision, and "is this page declining" is literally part of that decision, exactly like the lane guide's own `stale_visible_page` / `declining_with_demand` flags do. The leakage rule this protects against is different: once modeling starts in W05, `trend_direction` and `trend_pct` must never be used as a model *feature*, because at that point they'd be standing in for the label being predicted. No product-decision columns (`health_score`, `priority_score`, `action_type`) exist in this dataset at all, and no future-window (`*_last_30d`, `*_prev_30d`) columns were used in the rule — confirmed above.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all) — **run this yourself in Colab**; all numbers above were verified against the real starter dataset from this repo, so it should reproduce exactly
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.